# qGAN Experiment Tutorial

Use this notebook to run or load one selected experiment, run or load a battery, and analyze completed checkpoints.


## 1. Setup

Run this notebook from inside the repository or from `qgan/notebooks`. The cell below makes `qgan/src` importable without installing the package.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
qgan_dir = next(
    (
        candidate
        for root in (cwd, *cwd.parents)
        for candidate in (root, root / "qgan")
        if (candidate / "src" / "qgan_v2").is_dir()
    ),
    None,
)
if qgan_dir is None:
    raise RuntimeError("Could not find the qgan project directory.")

src_path = qgan_dir / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
repo_root = qgan_dir.parent
print({"qgan_dir": qgan_dir, "repo_root": repo_root})

{'qgan_dir': PosixPath('/home/benat/Qiskit-IBM/qgan'), 'repo_root': PosixPath('/home/benat/Qiskit-IBM')}


In [2]:
from qgan_v2.analysis import ResultsAnalysis, display_rows
from qgan_v2.experiments import BatteryExperiments, SingleExperiment, manage_runtime_account
from qgan_v2.implementations.registry import IMPLEMENTATIONS
from qgan_v2.visualization import get_visual_config, run_visualization

sorted(IMPLEMENTATIONS)

['manual_estimator', 'qml_torch', 'runtime_packed']

## 2. IBM Runtime Credentials

This cell is optional. Enable it only when you need to save or verify IBM Runtime credentials.


In [3]:
SAVE_RUNTIME_ACCOUNT = False
CHECK_RUNTIME_ACCOUNT = False

backend_count = manage_runtime_account(
    save=SAVE_RUNTIME_ACCOUNT,
    check=CHECK_RUNTIME_ACCOUNT,
)
if backend_count is not None:
    print("Available backends:", backend_count)

## 3. Single Experiment

The first YAML file in `configs/singles` is selected by default. Set `single_config_path` to choose another one.


In [4]:
single_config_path = repo_root / "qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/config.yaml"

single = SingleExperiment.select(
    single_config_path,
    fallback_directory=qgan_dir / "configs" / "singles",
)
single.summary()

{'config': '/home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/config.yaml',
 'run_id': 'base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0',
 'checkpoint': '/home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/training_data.pth',
 'checkpoint_exists': True}

### Run Or Load

Choose one boolean. Function arguments stay in the function call.


In [5]:
RUN_SINGLE_EXPERIMENT = False
state = single.run_or_load(run=RUN_SINGLE_EXPERIMENT, reset_data=False)
print("loaded state:", single.checkpoint)

loaded state: /home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/training_data.pth


### Inspect Checkpoint


In [6]:
print("current_epoch:", state.current_epoch)
print("best_eval:", state.metrics.best_eval())

current_epoch: 238
best_eval: 3.032940695344694


### Visualize


In [7]:
VISUALIZE_EXPERIMENT_RESULTS = False

if single.checkpoint.exists() and VISUALIZE_EXPERIMENT_RESULTS:
    run_visualization(
        single.config_path,
        get_visual_config({
            "draw_circuits": False,
            "draw_hardware_layout": True,
            "draw_probs": False,
            "draw_images": True,
            "draw_results": True,
        }),
    )

## 4. Battery Experiments

The first YAML file in `configs/batteries` is selected by default. Set `battery_path` to choose another one.


In [8]:
battery_path = repo_root / "qgan/configs/batteries/train/train_times_gpu.yaml"

battery = BatteryExperiments.select(
    battery_path,
    fallback_directory=qgan_dir / "configs" / "batteries",
)
print("battery:", battery.battery_path)

battery: /home/benat/Qiskit-IBM/qgan/configs/batteries/train/train_times_gpu.yaml


### Run Or Load

Set `RUN_BATTERY_EXPERIMENTS = True` to run the battery. Leave it `False` to load existing checkpoints. In both cases, `battery.states` is populated from checkpoint files.


In [9]:
RUN_BATTERY_EXPERIMENTS = False

battery.run_or_load(
    run=RUN_BATTERY_EXPERIMENTS,
    reset_data=False,
    reset_real_backend_info=False,
    stop_on_error=False,
    overwrite=False,
)

{'battery': '/home/benat/Qiskit-IBM/qgan/configs/batteries/train/train_times_gpu.yaml',
 'config_files': 267,
 'valid_config_files': 267,
 'invalid_config_files': 0,
 'loaded_states': 239,
 'missing_checkpoints': 28}

### Battery Summary


In [10]:
display_rows(battery.config_rows(limit=20))
remaining = len(battery.valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")

{'run_id': 'amp-qml_torch-q16-noiseless-PSR-aerCPU-rand0-seed0', 'done': False, 'loaded': False, 'config_file': '/home/benat/Qiskit-IBM/qgan/data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand0-seed0/config.yaml'}
{'run_id': 'amp-qml_torch-q16-noiseless-PSR-aerCPU-rand1-seed0', 'done': False, 'loaded': False, 'config_file': '/home/benat/Qiskit-IBM/qgan/data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand1-seed0/config.yaml'}
{'run_id': 'amp-qml_torch-q16-noiseless-PSR-aerGPU-rand0-seed0', 'done': False, 'loaded': False, 'config_file': '/home/benat/Qiskit-IBM/qgan/data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand0-seed0/config.yaml'}
{'run_id': 'amp-qml_torch-q16-noiseless-PSR-aerGPU-rand1-seed0', 'done': False, 'loaded': False, 'config_file': '/home/benat/Qiskit-IBM/qgan/data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand1-seed0/config.yaml'}
{'run_id': 'amp-qml_torch-q16-noiseless-REG-aerCPU-rand0-seed0', 'done': False, 'loaded': False, 'config_fil

## 5. Implementation Adapters

Quick view of implementations in the battery configs loaded above.


In [11]:
display_rows(
    battery.implementation_rows(project_directory=qgan_dir, limit=20)
)
remaining = len(battery.valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")

{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand0-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'PSR'}
{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand1-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'PSR'}
{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand0-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'PSR'}
{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand1-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'PSR'}
{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-REG-aerCPU-rand0-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'REG'}
{'config_file': 'data/train/times/amp-qml_torch-q16-noiseless-REG-aerCPU-rand1-seed0/config.yaml', 'implementation': 'qml_torch', 'preset': 'amp', 'gradient': 'REG'}
{'co